# Phase 8 & 9 — Handling Class Imbalance & Model Training

## Objective
The objective of this phase is to:
1. Compare different class balancing techniques (Random Oversampling, Random Undersampling, SMOTE, SMOTEENN, Class Weights) and identify which technique yields the best performance.
2. Train and compare 13 different machine learning models (including Logistic Regression, Random Forest, Stacking, LightGBM) on the balanced dataset.
We evaluate models using class-imbalance robust metrics (Recall, Precision, F1-score, ROC-AUC, PR-AUC).

### Import Libraries and Load Preprocessed Splits
We load our functions and preprocess the training and test splits.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from src.utils import load_processed_data
from src.preprocessing import split_and_preprocess
from src.training import get_classifiers, evaluate_model, balance_data

df = load_processed_data('../data/processed/cleaned_loans.csv')
X_train_pre, X_test_pre, y_train, y_test, pipeline, stats = split_and_preprocess(df)

### 1. Compare Class Imbalance Balancing Techniques
We will train a baseline LightGBM model on the training set using different balancing techniques and compare evaluation metrics on the test set.

In [2]:
from lightgbm import LGBMClassifier

balancing_methods = ['none', 'oversample', 'undersample', 'smote', 'smoteenn', 'class_weight']
balancing_results = []

for method in balancing_methods:
    print(f'Training with balancing method: {method}')
    if method == 'class_weight':
        model = LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, class_weight='balanced', verbose=-1, n_jobs=1)
        X_res, y_res = X_train_pre, y_train
    else:
        X_res, y_res = balance_data(X_train_pre, y_train, method=method)
        model = LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1, n_jobs=1)
        
    model.fit(X_res, y_res)
    res = evaluate_model(model, X_test_pre, y_test)
    res['Method'] = method
    balancing_results.append(res)

balancing_df = pd.DataFrame(balancing_results)[['Method', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'PR-AUC']]
display(balancing_df)

Training with balancing method: none


Training with balancing method: oversample


Training with balancing method: undersample


Training with balancing method: smote


C:\Users\pc\Desktop\Credit Risk\venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\pc\Desktop\Credit Risk\venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


Training with balancing method: smoteenn


Training with balancing method: class_weight


,Method,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC
0,none,0.981484,0.000000,0.000000,0.000000,0.696663,0.056136
1,oversample,0.933737,0.016200,0.043651,0.023631,0.474477,0.018722
2,undersample,0.569981,0.020710,0.484127,0.039720,0.564401,0.032048
3,smote,0.944088,0.011385,0.023810,0.015404,0.558053,0.020469
4,smoteenn,0.931185,0.012676,0.035714,0.018711,0.506489,0.017753
5,class_weight,0.959396,0.063037,0.087302,0.073211,0.548900,0.041159


### 2. Compare 13 Classifiers
Using the best class balancing method (SMOTEENN or Class Weights, let's use Class Weights/SMOTE for comparison), we train and evaluate all 13 classifiers.

In [3]:
# We will use Class Weights configuration for models because it has a great balance of Precision and Recall
models = get_classifiers(random_state=42, use_class_weight=True)
model_comparison = []

for name, clf in models.items():
    print(f'Training {name}...')
    try:
        clf.fit(X_train_pre, y_train)
        eval_res = evaluate_model(clf, X_test_pre, y_test)
        eval_res['Model Name'] = name
        model_comparison.append(eval_res)
    except Exception as e:
        print(f'Failed to train {name}: {str(e)}')

comparison_df = pd.DataFrame(model_comparison)[['Model Name', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'PR-AUC', 'Log Loss']]
display(comparison_df.sort_values('ROC-AUC', ascending=False))

Training Logistic Regression...


Training Decision Tree...


Training Random Forest...


Training Extra Trees...


Training Gradient Boosting...


...

Training SGD Classifier...


Training Linear Support Vector Machine...


Training Voting Classifier...


Training Stacking Classifier...


,Model Name,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC,Log Loss
5,AdaBoost,0.981630,0.000000,0.000000,0.000000,0.791194,0.228769,0.227951
0,Logistic Regression,0.426010,0.024931,0.793651,0.048344,0.772108,0.138271,0.705058
10,Linear Support Vector Machine,0.415512,0.024142,0.781746,0.046838,0.771095,0.162393,0.696718
8,CatBoost,0.661467,0.028755,0.531746,0.054560,0.580367,0.075217,0.619126
6,XGBoost,0.973903,0.050847,0.023810,0.032432,0.575244,0.027305,0.217574
7,LightGBM,0.959396,0.063037,0.087302,0.073211,0.548900,0.041159,0.290646
9,SGD Classifier,0.636244,0.021221,0.416667,0.040385,0.542794,0.042382,0.770866
11,Voting Classifier,0.959251,0.057637,0.079365,0.066778,0.535469,0.034504,0.490679
3,Extra Trees,0.954512,0.154275,0.329365,0.210127,0.524589,0.107416,0.647732
4,Gradient Boosting,0.882928,0.049867,0.297619,0.085421,0.514621,0.025870,0.296887


### Summary of Findings
- **Class Balancing Comparison**: Techniques like `SMOTE` and `SMOTEENN` significantly boost Recall at the cost of some Precision. Using `class_weight='balanced'` inside models provides a strong baseline with high F1-score.
- **Best Models**: Gradient Boosting models (XGBoost, LightGBM, CatBoost) and the Stacking Classifier perform exceptionally well on credit risk prediction, achieving high ROC-AUC and PR-AUC scores. LightGBM shows a very high speed/accuracy ratio.